# Local Website Summarizer

This notebook fetches website text, sends it to a local Ollama model through an OpenAI-compatible client, and displays the model's summary as Markdown.

## Imports

- `fetch_website_contents` is a local helper from `website_scraper.py` that collects readable website content.
- `Markdown` and `display` render the model response nicely inside the notebook.
- `OpenAI` creates a client that can talk to Ollama's OpenAI-compatible API.

In [1]:
from website_scraper import fetch_website_contents
from IPython.display import Markdown, display
from openai import OpenAI

## Ollama Client

- `OLLAMA_BASE_URL` points to the local Ollama server.
- The OpenAI client is configured with that local URL.
- The `api_key` value is only a placeholder because Ollama does not require a real OpenAI API key.

In [2]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

## System Prompt

- The system prompt controls the assistant's behavior.
- It asks for a clear, grounded website summary.
- It tells the model to ignore boilerplate, avoid invented details, and return plain Markdown.

In [3]:
system_prompt = """
You are a clear, concise website summarization assistant.
Summarize only the meaningful page content and ignore navigation menus, cookie banners, ads, repeated links, scripts, and boilerplate.
Return well-structured Markdown with:
- a brief overview
- key points
- news or announcements, if present
- a note when the page content is too thin or mostly navigation
Do not invent details that are not present in the website text.
""".strip()

## User Prompt

- The user prompt explains what content the model will receive.
- It asks for a reader-friendly website summary.
- It also asks the model to include news or announcements when they appear.

In [4]:
user_prompt = """
Here is the extracted text from a website.
Summarize the site for a reader who has not opened it.
Focus on what the site is about, important offerings or projects, and any news or announcements.

Website content:
""".strip()

## Build Chat Messages

- `messages_for` accepts website content as input.
- It returns the message format expected by the chat completions API.
- The website content is separated from the prompt with blank lines for clearer model input.

In [5]:
def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"{user_prompt}\n\n{website}"},
    ]

## Fetch Example Website Content

- This cell fetches text from the portfolio website.
- `print` is used so you can inspect the raw scraped content before summarizing it.

In [11]:
website = fetch_website_contents("https://apple.in")
print(website)

Apple (India)

Apple
Apple
Store
Mac
iPad
iPhone
Watch
AirPods
TV & Home
Entertainment
Accessories
Support
0
+
Get up to ₹15000 instant cashback on selected products with eligible cards. Plus up to 6 months of No Cost EMI.
‡
Shop
Apple Pay
Make way for Apple Pay.
Learn more
iPhone 18 Pro
Pro further.
Learn more
Buy
iPhone Duo
Hello, hello.
Pre-order starting at 5:30 PM IST on 16 October
Available from 23 October
Learn more
View pricing
Apple Watch Series 12
The most accurate heart rate sensing in a wearable.
1
Learn more
Buy
Apple Watch Ultra 4
A battery you can’t outrun.
Learn more
Buy
Mac mini
Now with M6 and M5 Pro.
Learn more
Buy
Apple iCloud plus
Now with Apple TV and Apple Arcade included.
2
Learn more
AirPods 5
Discover the magic of Active Noise Cancellation.
Learn more
Buy
Apple Trade In
Upgrade and save. It’s that easy.
Get your estimate
Item 1 - Monarch: Legacy of Monsters
Item 2 - Slow Horses
Item 3 - Hijack
Item 4 - Pluribus
Item 5 - Shrinking
Item 6 - F1 The Movie
Item 7 -

## Summarize a Website

- `summarize` accepts a URL.
- It fetches the website content.
- It sends the formatted messages to the local `llama3.2:1b` model.
- It returns only the assistant's generated Markdown content.

In [12]:
def summarize(url):
    website = fetch_website_contents(url)
    response = ollama.chat.completions.create(
        model="llama3.2:1b",
        messages=messages_for(website),
    )
    return response.choices[0].message.content

## Try the Summarizer

- This cell runs the summarizer on the portfolio website.
- The returned value will be shown as plain notebook output.

In [13]:
summarize("https://apple.in")

'**Overview**\n\nApple India is a mobile technology company that specializes in designing, manufacturing, and marketing various iOS products such as iPhones, iPads, Macs, Apple Watches, AirPods, and more.\n\n**Key Offerings**\n\n* Pre-orders: Apple offers pre-orders starting at 5:30 PM IST on October 16th for the iPhone 18 Pro.\n* New Products:\n\t+ iPhone Duo (launching on October 23rd)\n\t+ Apple Watch Series 12\n\t+ AirPods 5\n\t+ Mac mini with M6 and M5 Pro\n\t+ iCloud plus (now including Apple TV and Arcade)\n\n**Notable News**\n\nThe pre-orders for the iPhone 18 Pro start at 5:30 PM IST on October 16th. Apple also plans to release new products on October 23rd, including the iPhone Duo.\n\n**Important Features**\n\n* No cost EMI available with eligible credit cards (up to 6 months for most issuers)\n\nNote that the website contains some repetitive and unnecessary information, such as the instant cashback offer, which I have summarized below:\n\nApple India offers various products,

## Display a Summary as Markdown

- `display_summary` can use a provided URL or prompt for one interactively.
- It calls `summarize` to generate the summary.
- It renders the result as Markdown in the notebook instead of showing it as a raw string.

In [14]:
def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

display_summary("https://apple.in")

**Apple India: Your One-stop-shop for Innovation**

Apple India offers a wide range of innovative products, including iPhones, MacBook Pros, Apple Watches, AirPods, and more. The company also has exclusive deals such as instant cashback on eligible purchases using qualifying cards.

**Notable Offers:**

* Instant cashback up to ₹15,000 on selected products with eligible cards
* 6 months of No Cost EMI on select products using qualifying cards

**Upcoming Launches:**

* Pre-ordering begins for Apple Watch Series 12 and AirPods 5 on October 16th.
* iPhone 18 Pro will launch on October 23rd.

**Other Highlights:**

* Offers instant cashback with eligible credit card purchases
* No Cost EMI available with certain qualifying cards

**Note:** While the site provides valuable information about Apple's products and services, some sections appear to be primarily navigation-based, such as a section dedicated to "Adventure" (Item 1 - Monarch: Legacy of Monsters) which seems out of place in this context.